# 4 · make — decontamination contrast data

Pairs two MarinFold checkpoints protein by protein and breaks the difference out by how similar
each protein is to the *un-decontaminated* training set.

[#232](https://github.com/Open-Athena/MarinFold/issues/232) retrained the #199 recipe from scratch
on [#225](https://github.com/Open-Athena/MarinFold/issues/225)'s corpora, from which every
sequence matching any FoldBench chain at >= 30 % identity over >= 50 % of the shorter sequence was
removed. So the pair (`m2-p06`, `#199 cooldown`) is the same recipe with and without the eval
proteins' homologs in its training data, and the question this dataset answers is how much of the
cooldown's lead is leakage.

Read the strata carefully rather than as a verdict: proteins with no training homolog are also
much *harder* for both models, so a small absolute gap there is partly a floor effect. The
Spearman column is the check that does not have that problem.

CPU only; #245's published per-protein scores.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "4_contamination_contrast"
FOCUS = "#232 m2-p06 (decontaminated)"
BASELINE = "#199 cooldown (contaminated)"
EVAL_SETS = ["eval-val", "eval-test"]      # natural monomers; add eval-denovo for the designs
RANGE, METRIC = "all", "R"
BOOTSTRAP_DRAWS, BOOTSTRAP_SEED = 2000, 0
PARAMETERS = dict(focus=FOCUS, baseline=BASELINE, eval_sets=EVAL_SETS, range=RANGE, metric=METRIC,
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED)
PARAMETERS

In [ ]:
import numpy as np
import pandas as pd

inputs = figlib.Inputs()
targets = figlib.load_foldbench_universe(inputs)
scores = figlib.load_foldbench_scores(inputs)
scores = scores[(scores.range == RANGE) & (scores.cut == METRIC)]

units = targets[targets.eval_set.isin(EVAL_SETS)]
wide = (scores.pivot_table(index=["dataset", "stem"], columns="predictor", values="value")
              .reset_index())
paired = units.merge(wide, on=["dataset", "stem"], how="inner", validate="one_to_one")
missing = [name for name in (FOCUS, BASELINE) if name not in paired.columns]
if missing:
    raise SystemExit(f"{missing} were not scored on these sets; available: "
                     f"{sorted(scores.predictor.unique())}")
paired["delta"] = paired[FOCUS] - paired[BASELINE]
paired = paired.dropna(subset=[FOCUS, BASELINE])
print(f"{len(paired)} proteins scored by both")

In [ ]:
mean, low, high = figlib.bootstrap_mean(paired.delta.values, BOOTSTRAP_DRAWS, BOOTSTRAP_SEED)
overall = dict(group="all", n=len(paired), focus=paired[FOCUS].mean(),
               baseline=paired[BASELINE].mean(), delta=mean, ci_low=low, ci_high=high,
               focus_ahead_fraction=float((paired.delta > 0).mean()))
print(f"{FOCUS} − {BASELINE}: {mean:+.3f} [{low:+.3f}, {high:+.3f}] over {len(paired)} proteins; "
      f"focus ahead on {100 * overall['focus_ahead_fraction']:.0f}%")

rows = [overall]
for column in ("exp199_stratum", "is_viral"):
    frame = paired.copy()
    frame["group"] = frame[column].fillna("no_homolog").astype(str)
    for group, subset in frame.groupby("group"):
        mean, low, high = figlib.bootstrap_mean(subset.delta.values, BOOTSTRAP_DRAWS,
                                                BOOTSTRAP_SEED)
        rows.append(dict(group=f"{column}={group}" if column != "exp199_stratum" else group,
                         n=len(subset), focus=subset[FOCUS].mean(),
                         baseline=subset[BASELINE].mean(), delta=mean, ci_low=low, ci_high=high,
                         focus_ahead_fraction=float((subset.delta > 0).mean())))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# Does the gap track identity to the un-decontaminated training set, among proteins that have a
# homolog at all? A stratum trend can be a difficulty effect; a rank correlation cannot hide in one.
with_homolog = paired.dropna(subset=["exp199_best_identity"])
spearman = float(np.corrcoef(with_homolog.delta.rank(),
                             with_homolog.exp199_best_identity.rank())[0, 1])
print(f"spearman(delta, identity to #199 training set) = {spearman:+.3f} "
      f"over {len(with_homolog)} proteins with a homolog")

figlib.write_dataset(
    DATASET,
    notebook="4_make_contamination_contrast_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_protein.csv": lambda path: paired[
            ["dataset", "stem", "L", "eval_set", "is_viral", "exp199_stratum",
             "exp199_best_identity", FOCUS, BASELINE, "delta"]].to_csv(path, index=False),
    },
    extra={
        "contrast": {"focus": FOCUS, "baseline": BASELINE, "range": RANGE, "cut": METRIC},
        "result": {"paired_delta": overall["delta"], "ci": [overall["ci_low"], overall["ci_high"]],
                   "focus_ahead_fraction": overall["focus_ahead_fraction"],
                   "spearman_delta_vs_identity": spearman,
                   "n_with_homolog": int(len(with_homolog))},
    })